# 01 — Train

Single training notebook for all 13 thesis runs (Exp 1: 2 runs, Exp 2: 5 runs, Exp 3: 6 runs).

**Usage**: set `EXPERIMENT` in the registry cell, then run all cells top-down.

Hyperparameters fixed across runs (Table 3.3 in thesis): Adam, lr=1e-4, batch_size=64,
clip=4s × 16 frames × 112², RGB-normalised, FP16 AMP, horizontal flip p=0.5 where aug is on.


## 1. Experiment registry — pick which thesis run to reproduce

In [ ]:
# ---- BASE: hyperparameters that never change across thesis runs ----
BASE = dict(
    fps=25, clip_sec=4.0, clip_frames=16, clip_size=(112, 112),
    rgb_mean=(0.43216, 0.394666, 0.37645),
    rgb_std=(0.22803, 0.22145, 0.216989),
    optimizer="adam", lr=1e-4, batch_size=64, weight_decay=0.0,
    amp=True, label_smoothing=0.0,
    horizontal_flip_p=0.0,        # turned on where the experiment uses augmentation
    dropout=0.0, freeze_backbone=False,
    use_jitter=False, jitter_offsets=(-2.0, 0.0, +2.0),
    use_clip_cache=True,
    use_weighted_sampler=True,    # §3.3.2: weighted sampling for class imbalance (binary)
    min_neg_from_goal_s=12.0,
    seed=42,
)

# ---- Exp 1 (§4.1): binary R(2+1)D-18, jitter ON, 8 epochs ----
E1_BASE = {**BASE, "backbone": "r2plus1d_18", "task": "binary",
           "epochs": 8, "use_jitter": True,
           "neg_rand_per_goal": 4, "neg_per_no_goal_half": 12}

# ---- Exp 2 (§4.2): binary R(2+1)D-18 + hard negs, 10 epochs, 5 ablations (Table 4.5) ----
E2_BASE = {**E1_BASE, "epochs": 10,
           "use_hard_negs": True,
           "hard_neg_labels": ("shots on target", "shots off target", "penalty"),
           "hard_neg_per_goal": 1}

# ---- Exp 3 (§4.3): 4-class, no jitter, no dropout, no freeze, aug ON, 10 epochs ----
E3_BASE = {**BASE, "task": "4class", "epochs": 10,
           "backbone": "r2plus1d_18",  # overridden per preset
           "horizontal_flip_p": 0.5, "use_jitter": False,
           "neg_rand_per_goal": 4, "neg_per_no_goal_half": 12}

EXPERIMENTS = {
    # Experiment 1 — Tables 4.1, 4.2, 4.3
    "exp1_random_negs": {**E1_BASE, "use_hard_negs": False},
    "exp1_hard_negs":   {**E1_BASE, "use_hard_negs": True,
                         "hard_neg_labels": ("shots on target", "shots off target", "penalty"),
                         "hard_neg_per_goal": 1},

    # Experiment 2 — Table 4.5 (all use hard negs + R(2+1)D-18)
    "exp2_run1_dropout":            {**E2_BASE, "dropout": 0.4},
    "exp2_run2_dropout_freeze":     {**E2_BASE, "dropout": 0.4, "freeze_backbone": True},
    "exp2_run3_dropout_freeze_aug": {**E2_BASE, "dropout": 0.4, "freeze_backbone": True,
                                     "horizontal_flip_p": 0.5},
    "exp2_run4_aug":                {**E2_BASE, "horizontal_flip_p": 0.5},
    "exp2_run5_aug_no_jitter":      {**E2_BASE, "horizontal_flip_p": 0.5,
                                     "use_jitter": False},   # winner, val F1 = 0.868

    # Experiment 3 — Tables 4.6-4.8, 4.13 (six runs: 3 backbones × {1, 2} shots/goal)
    "exp3_r2plus1d_1shot": {**E3_BASE, "backbone": "r2plus1d_18",
                            "shots_on_per_goal": 1, "shots_off_per_goal": 1},
    "exp3_r2plus1d_2shot": {**E3_BASE, "backbone": "r2plus1d_18",
                            "shots_on_per_goal": 2, "shots_off_per_goal": 2},
    "exp3_r3d_1shot":      {**E3_BASE, "backbone": "r3d_18",
                            "shots_on_per_goal": 1, "shots_off_per_goal": 1},
    "exp3_r3d_2shot":      {**E3_BASE, "backbone": "r3d_18",
                            "shots_on_per_goal": 2, "shots_off_per_goal": 2},
    "exp3_mc3_1shot":      {**E3_BASE, "backbone": "mc3_18",
                            "shots_on_per_goal": 1, "shots_off_per_goal": 1},
    "exp3_mc3_2shot":      {**E3_BASE, "backbone": "mc3_18",
                            "shots_on_per_goal": 2, "shots_off_per_goal": 2},
}

# ============================ EDIT THIS LINE =================================
EXPERIMENT = "exp2_run5_aug_no_jitter"
# =============================================================================

cfg = dict(EXPERIMENTS[EXPERIMENT])
print(f"EXPERIMENT: {EXPERIMENT}")
for k, v in cfg.items():
    print(f"  {k:24s} = {v}")


## 2. Imports + derived paths

In [ ]:
import os, json, random, hashlib, pickle
from pathlib import Path
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms.v2 as Tv2
from torchvision.models.video import (
    r2plus1d_18, R2Plus1D_18_Weights,
    r3d_18,      R3D_18_Weights,
    mc3_18,      MC3_18_Weights,
)
from sklearn.metrics import (
    f1_score, precision_score, recall_score, confusion_matrix,
)
from tqdm.auto import tqdm

# ---- Paths ----
PROJECT_DIR = Path("/home/jinny/aspotting")
DATA_DIR    = PROJECT_DIR / "dataset"

# Cache key: (clip_sec, jitter) — binary and 4-class can share when geometry matches
cache_tag      = f"{cfg['clip_sec']:g}s" + ("_jitter" if cfg["use_jitter"] else "_center")
CLIP_CACHE_DIR = PROJECT_DIR / "clip_cache" / cache_tag

# Manifest key: task + cache_tag + negatives composition
neg_tag        = "hardneg" if cfg.get("use_hard_negs") else "randneg"
shots_tag      = f"_shots{cfg.get('shots_on_per_goal', 0)}-{cfg.get('shots_off_per_goal', 0)}" if cfg["task"] == "4class" else ""
MANIFEST_DIR   = PROJECT_DIR / "manifests" / f"{cfg['task']}_{cache_tag}_{neg_tag}{shots_tag}"

CKPT_DIR       = PROJECT_DIR / "checkpoints" / EXPERIMENT

for d in (CLIP_CACHE_DIR, MANIFEST_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MEAN   = np.array(cfg["rgb_mean"], dtype=np.float32)
STD    = np.array(cfg["rgb_std"],  dtype=np.float32)

print(f"Device         : {DEVICE}")
print(f"Cache dir      : {CLIP_CACHE_DIR}")
print(f"Manifest dir   : {MANIFEST_DIR}")
print(f"Checkpoint dir : {CKPT_DIR}")


## 2b. Download SoccerNet splits (skips already-downloaded files)

In [ ]:
DOWNLOAD_IF_MISSING = True   # set False to skip; SoccerNet downloader skips existing files anyway
SPLITS_TO_FETCH     = ['train', 'valid']

if DOWNLOAD_IF_MISSING:
    try:
        from SoccerNet.Downloader import SoccerNetDownloader
    except ImportError:
        raise ImportError("pip install SoccerNet")
    nv_pw = os.environ.get("NV_PASSWORD")
    if not nv_pw:
        raise RuntimeError(
            "Set NV_PASSWORD in env (register at https://www.soccer-net.org/data). "
            "Example: export NV_PASSWORD=...   or set DOWNLOAD_IF_MISSING=False to skip."
        )
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    d = SoccerNetDownloader(LocalDirectory=str(DATA_DIR))
    d.password = nv_pw
    print(f"Downloading SoccerNet splits {SPLITS_TO_FETCH} into {DATA_DIR} …")
    d.downloadGames(files=["Labels-v2.json"],                  split=SPLITS_TO_FETCH)
    d.downloadGames(files=["1_224p.mkv", "2_224p.mkv"],         split=SPLITS_TO_FETCH)
    print("Download step done (existing files were skipped).")


## 3. SoccerNet official splits (local games only)

In [ ]:
try:
    from SoccerNet.Downloader import getListGames
except ImportError:
    raise ImportError("pip install SoccerNet")

all_splits = {}
for split in ("train", "valid", "test"):
    games = getListGames(split)
    local = [g for g in games if (DATA_DIR / g / "Labels-v2.json").exists()]
    all_splits[split] = local
    print(f"{split:5s}  official={len(games):3d}  local={len(local):3d}")


## 4. Annotation parser + sample builder (binary or 4-class)

In [ ]:
SHOT_ON_LABELS  = {"shots on target", "penalty"}    # class 2 in 4-class (penalty → shots_on)
SHOT_OFF_LABELS = {"shots off target"}              # class 1 in 4-class
GOAL_LABEL      = "goal"                            # class 1 binary, class 3 in 4-class

CID_GOAL_4   = 3
CID_SHOT_ON  = 2
CID_SHOT_OFF = 1
CID_BG       = 0

if cfg["task"] == "binary":
    CLASS_NAMES = ["background", "goal"]
    NUM_CLASSES = 2   # head uses 1 sigmoid output for binary
else:
    CLASS_NAMES = ["background", "shots_off", "shots_on", "goal"]
    NUM_CLASSES = 4


def parse_annotations(game_rel_path):
    """Returns {half: [(seconds, class_id_or_label), ...]} for the active task."""
    with open(DATA_DIR / game_rel_path / "Labels-v2.json") as f:
        data = json.load(f)
    events = defaultdict(list)
    for ann in data["annotations"]:
        label = ann.get("label", "").strip().lower()
        half_str, _ = ann["gameTime"].split(" - ")
        half  = int(half_str)
        pos_s = int(ann["position"]) / 1000.0
        if cfg["task"] == "binary":
            if label == GOAL_LABEL:
                events[half].append((pos_s, "goal"))
            elif cfg.get("use_hard_negs") and label in set(cfg["hard_neg_labels"]):
                events[half].append((pos_s, label))   # keep specific label for breakdown
        else:
            if label == GOAL_LABEL:
                events[half].append((pos_s, CID_GOAL_4))
            elif label in SHOT_ON_LABELS:
                events[half].append((pos_s, CID_SHOT_ON))
            elif label in SHOT_OFF_LABELS:
                events[half].append((pos_s, CID_SHOT_OFF))
    return dict(events)


def far_from_all(t, times, min_gap):
    return all(abs(t - x) >= min_gap for x in times)


def build_samples_binary(game_list, is_train):
    """Binary: (video_path, center_sec, label[0|1], neg_type)."""
    samples = []
    for game in tqdm(game_list, desc="Building binary samples"):
        events = parse_annotations(game)
        for half in (1, 2):
            vid = DATA_DIR / game / f"{half}_224p.mkv"
            if not vid.exists():
                continue
            cap = cv2.VideoCapture(str(vid))
            fps_v = cap.get(cv2.CAP_PROP_FPS) or cfg["fps"]
            total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()
            dur = total / fps_v
            if dur < cfg["clip_sec"] + 1:
                continue
            lo, hi = cfg["clip_sec"]/2, dur - cfg["clip_sec"]/2

            half_evs = events.get(half, [])
            goals    = [t for t, lbl in half_evs if lbl == "goal"]
            hard     = [(t, lbl) for t, lbl in half_evs if lbl != "goal"]

            # Positives (+ jitter if enabled)
            jitters = cfg["jitter_offsets"] if (is_train and cfg["use_jitter"]) else (0.0,)
            for g in goals:
                for j in jitters:
                    samples.append((str(vid), float(np.clip(g + j, lo, hi)), 1.0, "goal"))

            # Hard negatives
            hard_cands = []
            if cfg.get("use_hard_negs"):
                hard_cands = [(t, lbl) for t, lbl in hard
                              if far_from_all(t, goals, cfg["min_neg_from_goal_s"])]
                n_hard = min(len(hard_cands), cfg["hard_neg_per_goal"] * max(1, len(goals)))
                if n_hard:
                    for t, lbl in random.sample(hard_cands, k=n_hard):
                        samples.append((str(vid), float(np.clip(t, lo, hi)), 0.0, lbl))

            # Random negatives
            n_rand = cfg["neg_rand_per_goal"] * len(goals) if goals else cfg["neg_per_no_goal_half"]
            forbid = list(goals) + [t for t, _ in hard_cands]
            added = attempts = 0
            while added < n_rand and attempts < n_rand * 40:
                attempts += 1
                t = random.uniform(lo, hi)
                if far_from_all(t, forbid, cfg["min_neg_from_goal_s"]):
                    samples.append((str(vid), t, 0.0, "rand"))
                    forbid.append(t)
                    added += 1
    return samples


def build_samples_4class(game_list, is_train):
    """4-class: (video_path, center_sec, class_id[0..3], event_type)."""
    samples = []
    for game in tqdm(game_list, desc="Building 4class samples"):
        events = parse_annotations(game)
        for half in (1, 2):
            vid = DATA_DIR / game / f"{half}_224p.mkv"
            if not vid.exists():
                continue
            cap = cv2.VideoCapture(str(vid))
            fps_v = cap.get(cv2.CAP_PROP_FPS) or cfg["fps"]
            total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()
            dur = total / fps_v
            if dur < cfg["clip_sec"] + 1:
                continue
            lo, hi = cfg["clip_sec"]/2, dur - cfg["clip_sec"]/2

            half_evs    = events.get(half, [])
            goals       = [t for t, c in half_evs if c == CID_GOAL_4]
            shots_on    = [t for t, c in half_evs if c == CID_SHOT_ON]
            shots_off   = [t for t, c in half_evs if c == CID_SHOT_OFF]
            all_event_t = [t for t, _ in half_evs]

            jitters = cfg["jitter_offsets"] if (is_train and cfg["use_jitter"]) else (0.0,)
            for g in goals:
                for j in jitters:
                    samples.append((str(vid), float(np.clip(g + j, lo, hi)), CID_GOAL_4, "goal"))

            on_cands  = [t for t in shots_on
                         if far_from_all(t, goals, cfg["min_neg_from_goal_s"])]
            n_on = min(len(on_cands), cfg["shots_on_per_goal"] * max(1, len(goals)))
            for t in random.sample(on_cands, k=n_on) if n_on else []:
                samples.append((str(vid), float(np.clip(t, lo, hi)), CID_SHOT_ON, "shots_on"))

            off_cands = [t for t in shots_off
                         if far_from_all(t, goals, cfg["min_neg_from_goal_s"])]
            n_off = min(len(off_cands), cfg["shots_off_per_goal"] * max(1, len(goals)))
            for t in random.sample(off_cands, k=n_off) if n_off else []:
                samples.append((str(vid), float(np.clip(t, lo, hi)), CID_SHOT_OFF, "shots_off"))

            n_rand = cfg["neg_rand_per_goal"] * len(goals) if goals else cfg["neg_per_no_goal_half"]
            forbid = list(all_event_t)
            added = attempts = 0
            while added < n_rand and attempts < n_rand * 40:
                attempts += 1
                t = random.uniform(lo, hi)
                if far_from_all(t, forbid, cfg["min_neg_from_goal_s"]):
                    samples.append((str(vid), t, CID_BG, "rand"))
                    forbid.append(t)
                    added += 1
    return samples


build_samples = build_samples_binary if cfg["task"] == "binary" else build_samples_4class
print("Sample builders defined")


## 5. Build / load manifest

In [ ]:
train_csv = MANIFEST_DIR / "train_clips.csv"
valid_csv = MANIFEST_DIR / "valid_clips.csv"

FORCE_REBUILD_MANIFEST = False
extra_col = "neg_type" if cfg["task"] == "binary" else "event_type"

if FORCE_REBUILD_MANIFEST or not (train_csv.exists() and valid_csv.exists()):
    random.seed(cfg["seed"])
    train_samples = build_samples(all_splits["train"], is_train=True)
    valid_samples = build_samples(all_splits["valid"], is_train=False)
    cols = ["video_path", "center_sec", "label", extra_col]
    pd.DataFrame(train_samples, columns=cols).to_csv(train_csv, index=False)
    pd.DataFrame(valid_samples, columns=cols).to_csv(valid_csv, index=False)
    print(f"Manifests written to {MANIFEST_DIR}")

train_df = pd.read_csv(train_csv)
valid_df = pd.read_csv(valid_csv)

print(f"\nTrain: {len(train_df):,} clips")
for cid in sorted(train_df.label.unique()):
    n = int((train_df.label == cid).sum())
    name = CLASS_NAMES[int(cid)] if cfg["task"] == "4class" else ("goal" if cid == 1.0 else "background")
    print(f"  label={cid}  ({name:12s}): {n:,}")
print(f"Valid: {len(valid_df):,} clips")


## 6. Clip cache builder (run once per cache_tag, then it's a no-op)

In [ ]:
def read_clip(video_path, center_sec):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None
    fps_v = cap.get(cv2.CAP_PROP_FPS) or cfg["fps"]
    n_frames = cfg["clip_frames"]
    clip_sec = cfg["clip_sec"]
    start_sec = max(0.0, center_sec - clip_sec/2)
    start_frame = int(start_sec * fps_v)
    step = max(1, int(clip_sec * fps_v / n_frames))
    frames = []
    for i in range(n_frames):
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame + i*step)
        ok, frame = cap.read()
        if not ok: break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, cfg["clip_size"])
        frames.append(frame)
    cap.release()
    while len(frames) < n_frames:
        frames.append(frames[-1] if frames else np.zeros((*cfg["clip_size"], 3), np.uint8))
    arr = np.stack(frames).astype(np.float32) / 255.0
    arr = (arr - MEAN) / STD
    return torch.from_numpy(arr).permute(3, 0, 1, 2)   # (C, T, H, W)


def clip_cache_path(video_path, center_sec):
    """Hash key: (video_path, center_sec). Class-agnostic — shared across tasks."""
    key = f"{video_path}_{center_sec:.4f}"
    h = hashlib.md5(key.encode()).hexdigest()
    return CLIP_CACHE_DIR / f"{h}.npy"


BUILD_CACHE = True  # safe to leave True — skips existing files

if BUILD_CACHE and cfg["use_clip_cache"]:
    for df, name in [(train_df, "train"), (valid_df, "valid")]:
        skipped = extracted = failed = 0
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"caching {name}"):
            out = clip_cache_path(row.video_path, float(row.center_sec))
            if out.exists():
                skipped += 1; continue
            clip = read_clip(row.video_path, float(row.center_sec))
            if clip is None:
                failed += 1; continue
            np.save(str(out), clip.numpy().astype(np.float16))
            extracted += 1
        print(f"  {name}: extracted={extracted}  skipped={skipped}  failed={failed}")

    n_cached = len(list(CLIP_CACHE_DIR.glob("*.npy")))
    size_gb  = sum(f.stat().st_size for f in CLIP_CACHE_DIR.glob("*.npy")) / 1e9
    print(f"\nCache: {n_cached:,} files  ({size_gb:.1f} GB)  at {CLIP_CACHE_DIR}")


## 7. Dataset + DataLoader (with optional weighted sampler)

In [ ]:
class ClipDataset(Dataset):
    def __init__(self, df, transform=None, task="binary"):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.task = task

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        clip = None
        if cfg["use_clip_cache"]:
            cache = clip_cache_path(row.video_path, float(row.center_sec))
            if cache.exists():
                clip = torch.from_numpy(np.load(str(cache)).astype(np.float32))
        if clip is None:
            clip = read_clip(row.video_path, float(row.center_sec))
            if clip is None:
                clip = torch.zeros(3, cfg["clip_frames"], *cfg["clip_size"])
        if self.transform is not None:
            clip = self.transform(clip)
        if self.task == "binary":
            return clip, torch.tensor(float(row.label), dtype=torch.float32)
        return clip, torch.tensor(int(row.label), dtype=torch.long)


train_tf = Tv2.RandomHorizontalFlip(p=cfg["horizontal_flip_p"]) if cfg["horizontal_flip_p"] > 0 else None

train_ds = ClipDataset(train_df, transform=train_tf, task=cfg["task"])
valid_ds = ClipDataset(valid_df, transform=None,     task=cfg["task"])

# Weighted random sampling (§3.3.2) — used in binary mode to balance pos/neg mini-batches.
if cfg["task"] == "binary" and cfg.get("use_weighted_sampler", True):
    class_counts = train_df["label"].value_counts().to_dict()
    weights = train_df["label"].map(lambda l: 1.0 / class_counts[l]).to_numpy()
    sampler = WeightedRandomSampler(weights=torch.from_numpy(weights).double(),
                                    num_samples=len(train_df), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], sampler=sampler,
                              num_workers=4, pin_memory=True, drop_last=True)
    print("Train sampler: WeightedRandomSampler")
else:
    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True,
                              num_workers=4, pin_memory=True, drop_last=True)
    print("Train sampler: shuffle")

valid_loader = DataLoader(valid_ds, batch_size=cfg["batch_size"], shuffle=False,
                          num_workers=4, pin_memory=True)

print(f"Train batches: {len(train_loader)}  Valid batches: {len(valid_loader)}")
print(f"Augmentation: {('horizontal flip p=' + str(cfg['horizontal_flip_p'])) if train_tf else 'OFF'}")


## 8. Model + loss + optimizer

In [ ]:
BACKBONES = {
    "r2plus1d_18": (r2plus1d_18, R2Plus1D_18_Weights.KINETICS400_V1),
    "r3d_18":      (r3d_18,      R3D_18_Weights.KINETICS400_V1),
    "mc3_18":      (mc3_18,      MC3_18_Weights.KINETICS400_V1),
}

factory, weights = BACKBONES[cfg["backbone"]]
model = factory(weights=weights)

if cfg["freeze_backbone"]:
    for name, p in model.named_parameters():
        if not (name.startswith("layer4") or name.startswith("fc")):
            p.requires_grad = False

out_features = 1 if cfg["task"] == "binary" else NUM_CLASSES
model.fc = nn.Sequential(
    nn.Dropout(p=cfg["dropout"]),
    nn.Linear(model.fc.in_features, out_features),
)
model = model.to(DEVICE)

if cfg["task"] == "binary":
    n_pos = int((train_df.label == 1.0).sum())
    n_neg = int((train_df.label == 0.0).sum())
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    print(f"pos_weight: {pos_weight.item():.3f}")
else:
    counts = train_df["label"].value_counts().sort_index()
    raw_w  = 1.0 / counts.values.astype(np.float32)
    class_weights = torch.tensor(raw_w / raw_w.mean(),
                                 dtype=torch.float32, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights,
                                    label_smoothing=cfg["label_smoothing"])
    print("class_weights:", {CLASS_NAMES[i]: round(float(w), 3)
                              for i, w in enumerate(class_weights.cpu().tolist())})

if cfg["optimizer"] == "adamw":
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                  lr=cfg["lr"], weight_decay=cfg["weight_decay"])
else:
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                                 lr=cfg["lr"], weight_decay=cfg["weight_decay"])

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["epochs"])
scaler    = torch.amp.GradScaler(DEVICE) if cfg["amp"] else None

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Backbone: {cfg['backbone']}  |  out_features: {out_features}")
print(f"Params  : {trainable:,} trainable / {total:,} total")


## 9. Training loop (W&B optional — set WANDB_API_KEY env var to enable)

In [ ]:
USE_WANDB = bool(os.environ.get("WANDB_API_KEY"))
if USE_WANDB:
    import wandb
    wandb.login(key=os.environ["WANDB_API_KEY"])
    run = wandb.init(
        project=os.environ.get("WANDB_PROJECT", "aspotting"),
        name=f"{EXPERIMENT}_{cfg['backbone']}",
        config={**cfg, "experiment": EXPERIMENT,
                "n_train_clips": len(train_df),
                "n_valid_clips": len(valid_df)},
        reinit=True,
    )


def run_epoch(loader, train):
    model.train() if train else model.eval()
    total_loss = total = 0
    all_preds, all_labels = [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for clips, labels in tqdm(loader, leave=False):
            clips  = clips.float().to(DEVICE)
            labels = labels.to(DEVICE)
            with torch.amp.autocast(DEVICE, enabled=cfg["amp"]):
                logits = model(clips)
                if cfg["task"] == "binary":
                    logits = logits.squeeze(1)
                    loss   = criterion(logits, labels)
                    preds  = (torch.sigmoid(logits) >= 0.5).float()
                else:
                    loss   = criterion(logits, labels)
                    preds  = torch.argmax(logits, dim=1)
            if train:
                optimizer.zero_grad()
                if scaler is not None:
                    scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
                else:
                    loss.backward(); optimizer.step()
            total_loss += loss.item() * len(labels)
            total      += len(labels)
            all_preds.extend(preds.detach().cpu().tolist())
            all_labels.extend(labels.detach().cpu().tolist())
    avg_loss = total_loss / total
    if cfg["task"] == "binary":
        p = precision_score(all_labels, all_preds, zero_division=0)
        r = recall_score(all_labels, all_preds, zero_division=0)
        f = f1_score(all_labels, all_preds, zero_division=0)
        return {"loss": avg_loss, "precision": p, "recall": r, "f1": f,
                "preds": all_preds, "labels": all_labels}
    else:
        macro = dict(
            macro_f1=f1_score(all_labels, all_preds, average="macro", zero_division=0),
            macro_p =precision_score(all_labels, all_preds, average="macro", zero_division=0),
            macro_r =recall_score(all_labels, all_preds, average="macro", zero_division=0),
        )
        per_f1 = f1_score(all_labels, all_preds, average=None,
                          labels=list(range(NUM_CLASSES)), zero_division=0)
        return {"loss": avg_loss, **macro, "per_f1": per_f1.tolist(),
                "preds": all_preds, "labels": all_labels}


run_dir = CKPT_DIR / datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir.mkdir(parents=True, exist_ok=True)
best_ckpt   = run_dir / "best.pt"
latest_ckpt = run_dir / "latest.pt"
best_val_f1 = 0.0
print(f"Checkpoint dir: {run_dir}")

for epoch in range(1, cfg["epochs"] + 1):
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(valid_loader, train=False)
    scheduler.step()

    if cfg["task"] == "binary":
        print(f"Epoch {epoch:02d}/{cfg['epochs']}  "
              f"train loss={tr['loss']:.4f} F1={tr['f1']:.3f}  |  "
              f"val loss={va['loss']:.4f} F1={va['f1']:.3f} P={va['precision']:.3f} R={va['recall']:.3f}")
        cur_f1 = va["f1"]
        log = {"train/loss": tr["loss"], "train/f1": tr["f1"],
               "val/loss":   va["loss"], "val/f1":   va["f1"],
               "val/precision": va["precision"], "val/recall": va["recall"]}
    else:
        print(f"Epoch {epoch:02d}/{cfg['epochs']}  "
              f"train loss={tr['loss']:.4f} macroF1={tr['macro_f1']:.3f}  |  "
              f"val loss={va['loss']:.4f} macroF1={va['macro_f1']:.3f}")
        per_str = "  ".join(f"{CLASS_NAMES[i]}={va['per_f1'][i]:.3f}" for i in range(NUM_CLASSES))
        print(f"  val per-class F1: {per_str}")
        cur_f1 = va["macro_f1"]
        log = {"train/loss": tr["loss"], "train/macro_f1": tr["macro_f1"],
               "val/loss":   va["loss"], "val/macro_f1":   va["macro_f1"]}
        for i, n in enumerate(CLASS_NAMES):
            log[f"val/f1_{n}"] = va["per_f1"][i]

    if USE_WANDB:
        wandb.log(log, step=epoch)

    torch.save({"epoch": epoch, "model": model.state_dict(),
                "optim": optimizer.state_dict(), "sched": scheduler.state_dict(),
                "val_f1": cur_f1, "best_val_f1": best_val_f1,
                "experiment": EXPERIMENT, "cfg": cfg}, latest_ckpt)
    if cur_f1 > best_val_f1:
        best_val_f1 = cur_f1
        torch.save(model.state_dict(), best_ckpt)
        print(f"  -> saved best  (val F1={best_val_f1:.4f})")

if USE_WANDB:
    run.finish()

print(f"\nDone. Best val F1: {best_val_f1:.4f}")
print(f"Best checkpoint: {best_ckpt}")
